In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Define features and target variable
feature = [
    "Stop:Station code", 
    "Hour",
    "DayOfWeek_sin",
    "DayOfWeek_cos", 
    "Month_sin", 
    "Month_cos", 
    "IsWeekend", 
    "RushHour",
    "StationTraffic",
    "Temperature",
    "Humidity",
    "Air Pressure",
    "Horizontal Visibility",
    "Cloud Cover",
    "Precipitation Amount",
    "Precipitation Duration",
    "Hourly Mean Wind Speed",
    "Max Wind Speed",
    "Fog",
    "Rainfall",
    "Snowfall",
    "Thunder",
    "Hail",
    "Service:Type"
    ]

target = "is_cancelled"

In [3]:
# Define file paths for each dataset split
files = {
    "Train": "train.csv",
    "Validation": "validation.csv",
    "Test": "test.csv"
}

# Read and process each dataset split in chunks
for split_name, file_path in files.items():
    print(f"\n{split_name} chunks:")
    total_rows = 0

    for i, chunk in enumerate(
        pd.read_csv(file_path, chunksize=1000000),
        start=1
    ):
        X_chunk = chunk[feature]
        y_chunk = chunk[target]
        total_rows += len(chunk)

        print(
            f"{split_name} chunk {i}: {chunk.shape}"
        )

    print(f"{split_name} total rows read: {total_rows:,}")


Train chunks:
Train chunk 1: (1000000, 26)
Train chunk 2: (1000000, 26)
Train chunk 3: (1000000, 26)
Train chunk 4: (1000000, 26)
Train chunk 5: (1000000, 26)
Train chunk 6: (1000000, 26)
Train chunk 7: (1000000, 26)
Train chunk 8: (1000000, 26)
Train chunk 9: (1000000, 26)
Train chunk 10: (1000000, 26)
Train chunk 11: (1000000, 26)
Train chunk 12: (1000000, 26)
Train chunk 13: (1000000, 26)
Train chunk 14: (1000000, 26)
Train chunk 15: (1000000, 26)
Train chunk 16: (1000000, 26)
Train chunk 17: (1000000, 26)
Train chunk 18: (1000000, 26)
Train chunk 19: (1000000, 26)
Train chunk 20: (1000000, 26)
Train chunk 21: (1000000, 26)
Train chunk 22: (1000000, 26)
Train chunk 23: (1000000, 26)
Train chunk 24: (1000000, 26)
Train chunk 25: (1000000, 26)
Train chunk 26: (1000000, 26)
Train chunk 27: (1000000, 26)
Train chunk 28: (1000000, 26)
Train chunk 29: (1000000, 26)
Train chunk 30: (1000000, 26)
Train chunk 31: (1000000, 26)
Train chunk 32: (1000000, 26)
Train chunk 33: (1000000, 26)
Trai

In [4]:
usecols = feature + [target]

train_df = pd.read_csv("train.csv", usecols=usecols)
validation_df = pd.read_csv("validation.csv", usecols=usecols)
test_df = pd.read_csv("test.csv", usecols=usecols)

for df in (train_df, validation_df, test_df):
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    df[float_cols] = df[float_cols].astype("float32")
    df[int_cols] = df[int_cols].astype("int32")

train_df = pd.get_dummies(
    train_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
validation_df = pd.get_dummies(
    validation_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
test_df = pd.get_dummies(
    test_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)

validation_df = validation_df.reindex(columns=train_df.columns, fill_value=0)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

X_train, y_train = train_df.drop(columns=[target]).astype("float32"), train_df[target].astype("int8")
X_val, y_val = validation_df.drop(columns=[target]).astype("float32"), validation_df[target].astype("int8")
X_test, y_test = test_df.drop(columns=[target]).astype("float32"), test_df[target].astype("int8")

In [11]:
# Random Forest from existing prepared data (no re-reading CSV).
# Stratified sampling to maintain class distribution in the sample
from sklearn.model_selection import StratifiedShuffleSplit
sample_size = 500_000
sss = StratifiedShuffleSplit(n_splits=1, train_size=sample_size, random_state=42)
for strat_idx, _ in sss.split(X_train, y_train):
    X_train_sample = X_train.iloc[strat_idx].astype("float32").to_numpy(copy=False)
    y_train_sample = y_train.iloc[strat_idx].to_numpy(copy=False)

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sample, y_train_sample)
print(f"Random Forest trained on {len(y_train_sample):,} samples (stratified)")

# Evaluate on validation and test sets in chunks to avoid memory issues
def evaluate_split_rf(name, X, y):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for start in range(0, len(X), 1_000_000):
        X_chunk = X.iloc[start:start + 1_000_000]
        y_chunk = y.iloc[start:start + 1_000_000]

        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)
        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(rf.predict(X_chunk_np))
        y_prob_parts.append(rf.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)
    
    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)
    print("F2 Score:", round(f2, 4))
    print(f"\n{name}")
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("ROC-AUC:", round(roc_auc_score(y_true_all, y_prob_all), 4))

evaluate_split_rf("Validation", X_val, y_val)
evaluate_split_rf("Test", X_test, y_test)

# Feature importances
feature_cols = X_train.columns
importances = pd.Series(
    rf.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)
 
print("\nTop 20 Feature Importances:")
print(importances.head(20))

Random Forest trained on 500,000 samples (stratified)
F2 Score: 0.238

Validation
              precision    recall  f1-score   support

           0      0.955     0.737     0.832   9439441
           1      0.087     0.422     0.144    560559

    accuracy                          0.719  10000000
   macro avg      0.521     0.579     0.488  10000000
weighted avg      0.907     0.719     0.793  10000000

ROC-AUC: 0.6071
F2 Score: 0.1839

Test
              precision    recall  f1-score   support

           0      0.931     0.867     0.898   8426700
           1      0.117     0.214     0.151    691095

    accuracy                          0.818   9117795
   macro avg      0.524     0.541     0.525   9117795
weighted avg      0.869     0.818     0.841   9117795

ROC-AUC: 0.601

Top 20 Feature Importances:
Service:Type_Sprinter     0.133774
Service:Type_Intercity    0.080499
Air Pressure              0.077206
Temperature               0.064967
Max Wind Speed            0.057786
Month_